# 04 — Unsupervised NLP Topic Modelling

This notebook discovers latent themes in a document corpus using TF-IDF, NMF, and latent semantic analysis.


In [ ]:
import sys
from pathlib import Path

import pandas as pd
from sklearn.decomposition import NMF, TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer

sys.path.append(str(Path.cwd() / "src"))

from unsup_lab.data import make_document_corpus
from unsup_lab.reporting import top_terms_by_component


In [ ]:
documents = make_document_corpus(random_state=42)
documents.head()


In [ ]:
vectorizer = TfidfVectorizer(
    min_df=2,
    max_df=0.9,
    ngram_range=(1, 2),
)

x_tfidf = vectorizer.fit_transform(documents["text"])
terms = vectorizer.get_feature_names_out()
x_tfidf.shape


## NMF topic model

In [ ]:
n_topics = 4

nmf = NMF(n_components=n_topics, random_state=42, init="nndsvda", max_iter=500)
document_topics = nmf.fit_transform(x_tfidf)

components = pd.DataFrame(
    nmf.components_,
    columns=terms,
    index=[f"topic_{i}" for i in range(n_topics)],
)

top_terms_by_component(components, n_terms=12)


In [ ]:
documents_with_topics = documents.copy()
documents_with_topics["dominant_topic"] = document_topics.argmax(axis=1)
documents_with_topics.head(10)


## Latent Semantic Analysis

In [ ]:
svd = TruncatedSVD(n_components=4, random_state=42)
lsa_embeddings = svd.fit_transform(x_tfidf)

lsa_components = pd.DataFrame(
    svd.components_,
    columns=terms,
    index=[f"component_{i}" for i in range(4)],
)

top_terms_by_component(lsa_components, n_terms=12)


## Notes

Topic models are sensitive to preprocessing, document length, vocabulary overlap, and domain-specific terms. The discovered topics should be reviewed by domain experts before being used in a workflow.
